# 078 — RLHF, RLAIF y DPO

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** P = σ(1,2 − (−0,3)) = σ(1,5) ≈ 0,817; L = −ln 0,817 ≈ **0,202**.
Sumando +5,0 a ambas, la diferencia sigue siendo 1,5: P y L **no cambian**. La
recompensa está definida salvo constante aditiva: solo las diferencias informan.

**Ejercicio 2.** Δ_w = −8,0 − (−8,4) = +0,4; Δ_l = −6,0 − (−5,2) = −0,8.
Margen = 0,2·(0,4 − (−0,8)) = 0,24. L_DPO = −ln σ(0,24) = −ln 0,560 ≈ **0,580**.
El margen ya es positivo (el modelo va en la dirección correcta), así que el
gradiente empuja de forma moderada, no agresiva.

**Ejercicio 3.** Reward hacking clásico: la política encontró respuestas (largas,
probablemente aduladoras) que inflan r_φ sin ser mejores; una KL débil (β bajo) le
permitió alejarse mucho de la referencia. Mitigaciones: subir β / early stopping
por KL; reentrenar el RM con los nuevos ejemplos hackeados etiquetados; añadir
normalización por longitud o evals humanas periódicas como freno.

**Ejercicio 4.** Las preferencias son comparaciones ruidosas modeladas con
distribuciones (sigmoide de diferencias): la base probabilística es exactamente el
contenido del laboratorio.

In [ ]:
import math
sigma = lambda z: 1 / (1 + math.exp(-z))

# Ejercicio 1
P = sigma(1.2 - (-0.3))
print(f"P={P:.3f}  L={-math.log(P):.3f}")          # 0.817, 0.202
P2 = sigma((1.2 + 5) - (-0.3 + 5))
print("invariante a constantes:", abs(P - P2) < 1e-12)

# Ejercicio 2
beta = 0.2
adv_w = -8.0 - (-8.4)
adv_l = -6.0 - (-5.2)
margen = beta * (adv_w - adv_l)
print(f"margen={margen:.2f}  L_DPO={-math.log(sigma(margen)):.3f}")  # 0.24, 0.580

# Ejercicio 4
result = run_lab("probability", seed=78)
assert result["kind"] == "probability"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué entrenar el RM con comparaciones (Bradley–Terry) es más robusto que
   pedir puntuaciones absolutas 1–10, y qué información se pierde a cambio?
2. Si al subir β en RLHF el modelo deja de mejorar en las evals de preferencia,
   ¿qué compromiso estás observando y cómo decidirías el β adecuado?
3. DPO optimiza el mismo objetivo teórico que RLHF-PPO: ¿qué pierde en la práctica
   por ser offline (sin generar muestras nuevas durante el entrenamiento)?